In [1]:
!pip install transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/datasets/atharvjairath/empathetic-dialogues-facebook-ai/emotion-emotion_69k.csv"
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Situation', 'emotion', 'empathetic_dialogues', 'labels', 'Unnamed: 5', 'Unnamed: 6'],
        num_rows: 64636
    })
})


In [3]:
print(dataset['train'][0])

{'Unnamed: 0': 0, 'Situation': 'I remember going to the fireworks with my best friend. There was a lot of people, but it only felt like us in the world.', 'emotion': 'sentimental', 'empathetic_dialogues': 'Customer :I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people, we felt like the only people in the world.\nAgent :', 'labels': 'Was this a friend you were in love with, or just a best friend?', 'Unnamed: 5': None, 'Unnamed: 6': None}


In [4]:
dataset = dataset.remove_columns([
    "Unnamed: 0",
    "Unnamed: 5",
    "Unnamed: 6"
])

print(dataset["train"].column_names)


['Situation', 'emotion', 'empathetic_dialogues', 'labels']


In [5]:
def build_dialogues(example):

    user_text = example["empathetic_dialogues"]
    response = example["labels"]
    emotion = example["emotion"]

    text = f"Customer: {user_text}\nEmotion: {emotion}\nAgent: {response}"

    return {"text": text}

In [6]:
dataset = dataset.map(build_dialogues)

Map:   0%|          | 0/64636 [00:00<?, ? examples/s]

In [7]:
print(dataset["train"][0]["text"])

Customer: Customer :I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people, we felt like the only people in the world.
Agent :
Emotion: sentimental
Agent: Was this a friend you were in love with, or just a best friend?


In [8]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
def tokenize(example):

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [10]:
tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/64636 [00:00<?, ? examples/s]

In [11]:
tokenized_dataset = tokenized_dataset.remove_columns([
    "Situation",
    "emotion",
    "empathetic_dialogues",
    "labels",
    "text"
])

In [12]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 64636
    })
})


In [13]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [14]:
training_args = TrainingArguments(
    
    output_dir="./empathetic_chatbot",

    per_device_train_batch_size=8,

    num_train_epochs=3,

    logging_steps=200,

    learning_rate=5e-5,

    weight_decay=0.01,

    save_strategy="epoch",

    fp16=True
)

In [15]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    processing_class=tokenizer,

    data_collator=data_collator
)

In [16]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
200,2.671182
400,2.523991
600,2.484417
800,2.459886
1000,2.446659
1200,2.414783
1400,2.419295
1600,2.413956
1800,2.384387
2000,2.381803


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12120, training_loss=2.2756388749226484, metrics={'train_runtime': 3229.7995, 'train_samples_per_second': 60.037, 'train_steps_per_second': 3.753, 'total_flos': 6333441289224192.0, 'train_loss': 2.2756388749226484, 'epoch': 3.0})

In [17]:
trainer.save_model("empathetic_chatbot_model")

tokenizer.save_pretrained("empathetic_chatbot_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('empathetic_chatbot_model/tokenizer_config.json',
 'empathetic_chatbot_model/tokenizer.json')

In [18]:
chatbot = pipeline(
    "text-generation",
    model="empathetic_chatbot_model",
    tokenizer=tokenizer
)

NameError: name 'pipeline' is not defined

In [19]:
prompt = "Customer: I feel really anxious about my exams.\nAgent:"

response = chatbot(
    prompt,
    max_length=80,
    temperature=0.7,
    do_sample=True
)

print(response[0]["generated_text"])

NameError: name 'chatbot' is not defined